In [9]:
import requests
import pandas as pd

# 東京の緯度経度
latitude = 35.6895
longitude = 139.6917

# APIリクエストのURL
url = (
    f"https://api.open-meteo.com/v1/forecast"
    f"?latitude={latitude}&longitude={longitude}&forecast_days=8"
    f"&daily=temperature_2m_max,temperature_2m_mean,temperature_2m_min,"
    f"precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,"
    f"shortwave_radiation_sum,"
    f"pressure_msl_mean,"
    f"&timezone=Asia%2FTokyo"
)

# APIリクエストの送信
response = requests.get(url)

# レスポンスの確認とデータの処理
if response.status_code == 200:
    data = response.json()
    daily = data["daily"]

    # DataFrameの作成
    df_weather = pd.DataFrame({
        "日付": pd.to_datetime(daily["time"]),
        "最高気温(℃)": daily["temperature_2m_max"],
        "平均気温(℃)": daily["temperature_2m_mean"],
        "最低気温(℃)": daily["temperature_2m_min"],
        "降水量の合計(mm)": daily["precipitation_sum"],
        "最大風速(m/s)": [v / 3.6 for v in daily["wind_speed_10m_max"]],  # km/h → m/s
        "平均湿度(％)": daily["relative_humidity_2m_mean"],
        "合計全天日射量(MJ/㎡)": daily["shortwave_radiation_sum"],
        "平均現地気圧(hPa)": daily["pressure_msl_mean"]
    })

    # 「平均風速(m/s)」を「最大風速(m/s)」で代用
    df_weather["平均風速(m/s)"] = df_weather["最大風速(m/s)"]

    # 曜日を追加（月=0, 日=6）
    weekday_jp = {
    "Monday": "月曜日",
    "Tuesday": "火曜日",
    "Wednesday": "水曜日",
    "Thursday": "木曜日",
    "Friday": "金曜日",
    "Saturday": "土曜日",
    "Sunday": "日曜日"
    }

    df_weather["曜日_英語"] = df_weather["日付"].dt.day_name()
    df_weather["曜日"] = df_weather["曜日_英語"].map(weekday_jp)
    df_weather.drop(columns=["曜日_英語"], inplace=True)

    df_weather["曜日番号"] = df_weather["日付"].dt.weekday



    # 表示
    print(df_weather)

else:
    print("APIリクエストに失敗しました。ステータスコード:", response.status_code)

          日付  最高気温(℃)  平均気温(℃)  最低気温(℃)  降水量の合計(mm)  最大風速(m/s)  平均湿度(％)  \
0 2025-06-19     31.6     26.1     21.4         0.0   2.083333       81   
1 2025-06-20     30.2     25.7     22.0         0.0   1.972222       81   
2 2025-06-21     29.8     25.5     22.4         0.0   2.611111       82   
3 2025-06-22     30.7     26.4     22.5         0.0   8.333333       73   
4 2025-06-23     31.7     26.7     23.7         0.0   7.500000       74   
5 2025-06-24     26.8     25.1     23.7         0.9   4.888889       87   
6 2025-06-25     31.1     27.0     23.9         0.6   5.805556       79   
7 2025-06-26     31.0     27.4     24.3         5.7   3.222222       76   

   合計全天日射量(MJ/㎡)  平均現地気圧(hPa)  平均風速(m/s)   曜日  曜日番号  
0          24.49       1014.3   2.083333  木曜日     3  
1          23.81       1015.2   1.972222  金曜日     4  
2          23.63       1012.4   2.611111  土曜日     5  
3          27.37       1008.1   8.333333  日曜日     6  
4          24.72       1007.6   7.500000  月曜日     0  


In [10]:
# ライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score



In [11]:
# データの読み込みと前処理
df = pd.read_csv('df_merge (2).csv')

print("データ形状:", df.shape)
print("データの最初の5行:")
display(df.head())

# 日付の変換
df['日付'] = pd.to_datetime(df['日付'])
df['year'] = df['日付'].dt.year
df['month'] = df['日付'].dt.month
df['day'] = df['日付'].dt.day
df['weekday'] = df['日付'].dt.weekday

# ビール売上列の特定
beer_sales_columns = [
    'ペールエール(本)', 'ラガー(本)', 'IPA(本)',
    'ホワイトビール(本)', '黒ビール(本)', 'フルーツビール(本)'
]

# 基本天気特徴量
weather_features = [
    '平均気温(℃)', '降水量の合計(mm)', '合計全天日射量(MJ/㎡)',
    '平均風速(m/s)', '平均現地気圧(hPa)',
    '平均湿度(％)', '最高気温(℃)', '最低気温(℃)', '曜日番号'
]

# 天気概況のエンコード
le = LabelEncoder()
df['天気概況_encoded'] = le.fit_transform(df['天気概況(昼：06時～18時)'].fillna('未知'))
weather_features.append('天気概況_encoded')

# 時間特徴量の追加
weather_features.extend(['year', 'month', 'day', 'weekday'])

print(f"ビール売上対象: {beer_sales_columns}")
print(f"天気特徴量数: {len(weather_features)}")

データ形状: (314, 35)
データの最初の5行:


,日付,曜日,予約件数,予約人数,来客数,総杯数,売上合計(円),ペールエール(本),ペールエール(円),ラガー(本),...,平均蒸気圧(hPa),平均現地気圧(hPa),平均湿度(％),最大風速(m/s),最大瞬間風速(m/s),最高気温(℃),最低気温(℃),降雪量合計(cm),天気概況コード,曜日番号
0,2024-04-01,月,NaN,NaN,16,25,25300,6,6000,5,...,11.9,1004.7,73.0,6.3,11.4,19.7,11.8,0.0,0,0
1,2024-04-02,火,NaN,NaN,19,25,24600,6,6000,4,...,8.2,1013.7,50.0,6.1,10.9,20.6,8.4,0.0,9,1
2,2024-04-03,水,NaN,NaN,11,19,18500,5,5000,4,...,13.0,1009.9,83.0,3.9,6.7,16.6,12.3,0.0,28,2
3,2024-04-04,木,NaN,NaN,6,10,9500,2,2000,2,...,13.6,1005.1,79.0,4.7,7.7,19.9,11.3,0.0,14,3
4,2024-04-05,金,NaN,NaN,10,18,17100,3,3000,5,...,9.6,1017.0,71.0,4.9,9.3,15.1,9.0,0.0,20,4


ビール売上対象: ['ペールエール(本)', 'ラガー(本)', 'IPA(本)', 'ホワイトビール(本)', '黒ビール(本)', 'フルーツビール(本)']
天気特徴量数: 14


In [12]:
#高度な特徴量エンジニアリング
print("特徴量エンジニアリング開始...")

# 1. 温度関連特徴量
df['温度差'] = df['最高気温(℃)'] - df['最低気温(℃)']
df['温度変化率'] = df['温度差'] / (df['平均気温(℃)'] + 1e-6)

# 2. 快適度指数
df['快適度指数'] = (
    (df['平均気温(℃)'] - 20).abs() * -1 +  # 理想温度20度
    (df['平均湿度(％)'] - 60).abs() * -0.5 +  # 理想湿度60%
    df['降水量の合計(mm)'] * -2  # 降雨減点
)
# 将 weekday 编码为 Dummy 变量（one-hot）
# 先清理旧的 dummy 列
for i in range(7):
    col_name = f'曜日_{i}'
    if col_name in df.columns:
        df.drop(columns=col_name, inplace=True)

# 再重新添加 dummy
weekday_dummies = pd.get_dummies(df['weekday'], prefix='曜日')
df = pd.concat([df, weekday_dummies], axis=1)


# 3. 季節性特徴量
df['季節'] = df['month'].map({
    12: 0, 1: 0, 2: 0,  # 冬
    3: 1, 4: 1, 5: 1,   # 春
    6: 2, 7: 2, 8: 2,   # 夏
    9: 3, 10: 3, 11: 3  # 秋
})

# 4. 週末フラグ
df['週末フラグ'] = (df['weekday'] >= 5).astype(int)

# 5. 天気分類
good_weather = ['晴', '快晴', '薄曇']
df['良天気'] = df['天気概況(昼：06時～18時)'].apply(
    lambda x: 1 if any(w in str(x) for w in good_weather) else 0
)


def classify_rainfall(x):
    if x == 0:
        return 0  # 无雨
    elif x <= 10:
        return 1  # 小雨
    else:
        return 2  # 大雨

# 风速分类函数
def classify_wind(x):
    if x <= 3:
        return 0  # 轻风
    elif x <= 10:
        return 1  # 中风
    else:
        return 2  # 强风

# 应用分类
df["風速分類"] = df["最大風速(m/s)"].apply(classify_wind)
df["降水量分類"] = df["降水量の合計(mm)"].apply(classify_rainfall)
# 生成 dummy 变量
df = pd.get_dummies(df, columns=["降水量分類","風速分類"], prefix=["降水量","風速"], dtype=int)


# 改良された特徴量リスト
improved_features = weather_features + [
    '温度差', '温度変化率', '快適度指数', '季節','週末フラグ','降水量_0','降水量_1','降水量_2','良天気','風速_0','風速_1','風速_2','曜日_0','曜日_1','曜日_2','曜日_3','曜日_4','曜日_5'
]

print(f"基本特徴量数: {len(weather_features)}")
print(f"改良特徴量数: {len(improved_features)}")
print(f"追加特徴量: {[f for f in improved_features if f not in weather_features]}")
print("特徴量エンジニアリング完了 ✅")

特徴量エンジニアリング開始...
基本特徴量数: 14
改良特徴量数: 32
追加特徴量: ['温度差', '温度変化率', '快適度指数', '季節', '週末フラグ', '降水量_0', '降水量_1', '降水量_2', '良天気', '風速_0', '風速_1', '風速_2', '曜日_0', '曜日_1', '曜日_2', '曜日_3', '曜日_4', '曜日_5']
特徴量エンジニアリング完了 ✅


In [13]:
# ライブラリのインポート
from sklearn.ensemble import RandomForestRegressor

def train_rf_model(df, target_col, feature_cols, test_size=0.2, random_state=42):
    """
    RandomForestRegressorでモデルを訓練する関数
    """
    # データの準備
    data = df[[target_col] + feature_cols].dropna()
    if len(data) == 0:
        print(f"警告: {target_col} に有効なデータがありません")
        return None, None

    X = data[feature_cols]
    y = data[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # モデルの定義と訓練
    model = RandomForestRegressor(n_estimators=1000, random_state=random_state, n_jobs=-1)
    model.fit(X_train, y_train)

    # 評価
    y_pred_test = model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

    # 結果の表示
    print(f"{target_col} (Random Forest):")
    print(f"  テストR²: {test_r2:.4f}, テストRMSE: {test_rmse:.4f}, データ数: {len(X_test)}")

    return test_r2

# # 各ビールに対してRFモデルを訓練・評価
# print("\n--- RandomForestモデルの訓練開始 ---")
# rf_r2_scores = {}
# for beer_col in beer_sales_columns:
#     if beer_col in df.columns:
#         r2 = train_rf_model(df, beer_col, improved_features)
#         if r2 is not None:
#             rf_r2_scores[beer_col] = r2

# print("-" * 50)

# # 平均R2スコアを計算・表示
# if rf_r2_scores:
#     avg_rf_r2 = np.mean(list(rf_r2_scores.values()))
#     print(f"\n✅ RandomForestモデルの平均決定係数（R²）: {avg_rf_r2:.4f}")


In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def preprocess_weather_rows(df_weather):
    """
    Open-Meteoから取得した天気DataFrameに特徴量エンジニアリングを適用
    """
    df_weather = df_weather.copy()

    # 必須の時間情報
    df_weather['year'] = df_weather['日付'].dt.year
    df_weather['month'] = df_weather['日付'].dt.month
    df_weather['day'] = df_weather['日付'].dt.day
    df_weather['weekday'] = df_weather['日付'].dt.weekday
    df_weather['曜日番号'] = df_weather['weekday']

    # 特徴量作成
    df_weather['温度差'] = df_weather['最高気温(℃)'] - df_weather['最低気温(℃)']
    df_weather['温度変化率'] = df_weather['温度差'] / (df_weather['平均気温(℃)'] + 1e-6)
    df_weather['快適度指数'] = (
        (df_weather['平均気温(℃)'] - 20).abs() * -1 +
        (df_weather['平均湿度(％)'] - 60).abs() * -0.5 +
        df_weather['降水量の合計(mm)'] * -2
    )
    df_weather['季節'] = df_weather['month'].map({
        12: 0, 1: 0, 2: 0,
        3: 1, 4: 1, 5: 1,
        6: 2, 7: 2, 8: 2,
        9: 3, 10: 3, 11: 3
    })
    df_weather['週末フラグ'] = (df_weather['weekday'] >= 5).astype(int)
    df_weather['良天気'] = 0  # APIには天気概況がないので一律0

    # 降水・風速分類
    df_weather['風速分類'] = df_weather['最大風速(m/s)'].apply(classify_wind)
    df_weather['降水量分類'] = df_weather['降水量の合計(mm)'].apply(classify_rainfall)

    # ダミー変数化
    df_weather = pd.get_dummies(df_weather, columns=["降水量分類", "風速分類"], prefix=["降水量", "風速"], dtype=int)

    for i in range(7):
        df_weather[f'曜日_{i}'] = (df_weather['曜日番号'] == i).astype(int)

    # 欠損補完（学習用特徴量と合わせる）
    for col in improved_features:
        if col not in df_weather.columns:
            df_weather[col] = 0

    return df_weather

def train_all_beer_models(df, features):
    """
    各ビールごとにモデルを訓練して辞書で返す
    """
    models = {}
    for col in beer_sales_columns:
        data = df[[col] + features].dropna()
        if data.empty:
            continue
        X = data[features]
        y = data[col]
        model = RandomForestRegressor(n_estimators=1000, random_state=42, n_jobs=-1)
        model.fit(X, y)
        models[col] = model
    return models

def predict_beer_sales(models, df_weather_processed):
    """
    天気データに基づいて各ビールの売上を予測
    """
    predictions = df_weather_processed[['日付']].copy()
    for beer, model in models.items():
        predict_X = df_weather_processed[model.feature_names_in_]
        predictions[beer] = model.predict(predict_X).round(1)
    return predictions


In [15]:
# ステップ1: 天気データに特徴量エンジニアリングを適用
weather_for_prediction = preprocess_weather_rows(df_weather)

# ステップ2: 学習データからモデルを訓練
models = train_all_beer_models(df, improved_features)

# ステップ3: 予測
predicted_beer_sales = predict_beer_sales(models, weather_for_prediction)

# 結果表示
print("\n📈 今後1週間のビール売上予測：")
display(predicted_beer_sales)



📈 今後1週間のビール売上予測：


,日付,ペールエール(本),ラガー(本),IPA(本),ホワイトビール(本),黒ビール(本),フルーツビール(本)
0,2025-06-19,4.9,6.2,3.9,4.2,2.6,4.2
1,2025-06-20,9.7,9.1,6.8,6.6,3.5,5.2
2,2025-06-21,4.7,5.7,3.7,4.3,2.7,3.7
3,2025-06-22,4.6,6.1,3.4,4.7,2.4,3.7
4,2025-06-23,4.4,6.8,3.3,4.7,2.4,3.4
5,2025-06-24,4.3,5.4,3.5,3.5,2.0,2.2
6,2025-06-25,4.1,5.5,3.1,3.8,2.1,3.1
7,2025-06-26,3.8,4.7,3.1,3.4,2.0,2.2


In [19]:
# 明确啤酒列
beer_columns = ["ペールエール(本)", "ラガー(本)", "IPA(本)", "ホワイトビール(本)", "黒ビール(本)", "フルーツビール(本)"]

# 日期处理
predicted_beer_sales["日付"] = pd.to_datetime(predicted_beer_sales["日付"])
predicted_beer_sales["weekday"] = predicted_beer_sales["日付"].dt.weekday

# 初始化空表
results = []

# 检查月曜用的发货日（月0、火1、水2）是否都存在
if set([0, 1, 2]).issubset(set(predicted_beer_sales["weekday"])):
    monday_group = predicted_beer_sales[predicted_beer_sales["weekday"].isin([0, 1, 2])]
    monday_sum = monday_group[beer_columns].sum().to_frame(name="月曜用の出荷集計").T
    results.append(monday_sum)

# 检查木曜用的发货日（木3、金4、土5）是否都存在
if set([3, 4, 5]).issubset(set(predicted_beer_sales["weekday"])):
    thursday_group = predicted_beer_sales[predicted_beer_sales["weekday"].isin([3, 4, 5])]
    thursday_sum = thursday_group[beer_columns].sum().to_frame(name="木曜用の出荷集計").T
    results.append(thursday_sum)

# 合并输出（只输出能算的）
if results:
    shipment_summary = pd.concat(results)
    print("📦 発注用ビール出荷集計:")
    print(shipment_summary)
else:
    print("⚠️ 月曜用・木曜用いずれも発注日が不足しています。")


📦 発注用ビール出荷集計:
          ペールエール(本)  ラガー(本)  IPA(本)  ホワイトビール(本)  黒ビール(本)  フルーツビール(本)
月曜用の出荷集計       12.8    17.7     9.9        12.0      6.5         8.7
木曜用の出荷集計       23.1    25.7    17.5        18.5     10.8        15.3
